# Ingest Bitcoin Prices using River for Real-Time Processing

This notebook demonstrates how to ingest real-time Bitcoin price data using the CoinGecko API and perform online learning using the River library.

**Goals:**
- Stream live Bitcoin price data
- Use River for incremental training
- Maintain a rolling window of prices
- Extract lag features for prediction
- Visualize real-time predictions and model accuracy


In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Imports
This cell initializes the required packages and imports the helper functions from bitcoin_forecast_utils.py.

In [2]:
!pip install river
!pip install pytest
!pip install scikit-learn
!pip install matplotlib
!pip install requests
!pip install streamlit
!pip install numpy as np

ERROR: Could not find a version that satisfies the requirement as (from versions: none)
ERROR: No matching distribution found for as


In [3]:
import logging
# Import libraries in this section.
# Avoid imports like import *, from ... import ..., from ... import *, etc.
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import collections

import river
import river.linear_model
import river.tree
import river.metrics
import river.optim
from river import linear_model
from river import metrics
import bitcoin_forecast_utils
from bitcoin_forecast_utils import (
    get_bitcoin_price_with_retry,
    get_coin_ohlc,
    build_rolling_features
)
from collections import deque
from river import preprocessing
from river import tree
import requests
import pickle
import streamlit as st
from IPython.display import Image

# Initialize Model, Metric, Rolling Structures
This cell performs essential setup for real-time streaming:

- **Model Initialization**: Uses River's `StandardScaler` and `LinearRegression` in a pipeline to handle online feature normalization and regression.
- **Metric**: Initializes `MAE` (Mean Absolute Error) for evaluating prediction accuracy incrementally.
- **Rolling Window**: Creates a `deque` to maintain the most recent 5 Bitcoin prices for feature engineering.
- **Logging**: Prepares empty lists to store MAE values, predictions, and actual prices for post-streaming visualization and analysis.

In [4]:
# Initialize River model and metric
model = preprocessing.StandardScaler() | linear_model.LinearRegression()
metric = metrics.MAE()

# Rolling window to hold past prices (lagged features)
rolling_prices = deque(maxlen=5)

# Logs for analysis and plotting
mae_log = []
pred_log = []
true_log = []


##  Real-Time Streaming + Online Model Training
This block simulates **real-time model training** using cached OHLC close prices. Here's what each part does:

- **OHLC Fetch**: Retrieves 1 day of close prices from the CoinGecko API via `get_coin_ohlc(days=1)` to mimic a live stream.
- **Streaming Simulation**: Iterates through the close prices one-by-one as if they’re arriving in real time.
- **Rolling Window**: Appends each new price to a `deque`, maintaining only the latest 5 values.
- **Feature Engineering**: Constructs combined features using both price lags and OHLC-derived volatility indicators.
- **Model Prediction**: Makes a prediction using the current model.
- **Model Training**: If a prediction is available, the model is updated with the actual target using `learn_one()`.
- **Metric Update**: The MAE (Mean Absolute Error) is updated incrementally for evaluation.
- **Logging**: Records MAE, prediction, and actual values for later plotting.

This structure supports **streaming learning** with continuous model updates, performance tracking, and feature interaction using River.

In [5]:
# Simulate real-time from cached OHLC close prices
ohlc_df = get_coin_ohlc(days=1)
for step, price in enumerate(ohlc_df["close"].head(30)):
    rolling_prices.append(price)

    if len(rolling_prices) < rolling_prices.maxlen:
        continue

    #features = build_rolling_features(rolling_prices)
    ohlc_features_df = bitcoin_forecast_utils.extract_ohlc_features(ohlc_df)
    features = bitcoin_forecast_utils.build_combined_features(rolling_prices, ohlc_features_df)
    true_price = features["price_lag_0"]
    pred_price = model.predict_one(features)

    if pred_price is not None:
        model.learn_one(features, true_price)
        metric = metric.update(true_price, pred_price)

        mae_log.append(metric.get())
        pred_log.append(pred_price)
        true_log.append(true_price)


INFO:bitcoin_forecast_utils:Loading cached OHLC data



This log message is generated by the get_coin_ohlc() function from the bitcoin_forecast_utils.py module.

 What It Means:
- The notebook **did not fetch fresh OHLC data** from the CoinGecko API.
- Instead, it **loaded previously saved OHLC data** from a local CSV file (cached_ohlc.csv).
- This behavior is controlled by the caching mechanism to improve efficiency and reduce API calls.

 Why This Is Useful:
- **Faster Execution:** Avoids unnecessary API calls.
- **Rate Limit Protection:** Prevents hitting CoinGecko’s usage limits.
- **Reproducibility:** Ensures consistent results during testing or demonstration.

## Rolling MAE Over Time
This cell visualizes the **Mean Absolute Error (MAE)** of the model predictions over time.

In [11]:
def print_mae_log(mae_log):
    print("Rolling MAE values over time:")
    for i, mae in enumerate(mae_log):
        print(f"Step {i + 1}: MAE = {mae:.2f}")
print_mae_log(mae_log)

Rolling MAE values over time:
Step 1: MAE = 96855.00
Step 2: MAE = 95714.95
Step 3: MAE = 89141.24
Step 4: MAE = 83528.05
Step 5: MAE = 78829.00
Step 6: MAE = 76808.68
Step 7: MAE = 74891.99
Step 8: MAE = 72876.05
Step 9: MAE = 68400.12
Step 10: MAE = 62458.31
Step 11: MAE = 58905.17
Step 12: MAE = 56820.67
Step 13: MAE = 54546.07
Step 14: MAE = 52448.08
Step 15: MAE = 50358.64
Step 16: MAE = 47774.23
Step 17: MAE = 45241.53
Step 18: MAE = 42990.82
Step 19: MAE = 41158.18
Step 20: MAE = 39837.76
Step 21: MAE = 38637.61
Step 22: MAE = 37485.71
Step 23: MAE = 36434.07
Step 24: MAE = 35457.33
Step 25: MAE = 34477.72
Step 26: MAE = 33540.07


In [13]:
def display_mae_table(mae_log):
    df = pd.DataFrame({'Time Step': range(1, len(mae_log) + 1), 'MAE': mae_log})
    print(df.to_string(index=False))
display_mae_table(mae_log)

 Time Step          MAE
         1 96855.000000
         2 95714.950000
         3 89141.236797
         4 83528.050575
         5 78828.997522
         6 76808.676217
         7 74891.985520
         8 72876.054892
         9 68400.118720
        10 62458.310311
        11 58905.174779
        12 56820.672496
        13 54546.070211
        14 52448.076045
        15 50358.640613
        16 47774.232949
        17 45241.527480
        18 42990.819285
        19 41158.179171
        20 39837.764066
        21 38637.606189
        22 37485.706159
        23 36434.066924
        24 35457.332027
        25 34477.718699
        26 33540.068570


The MAE steadily decreases over time, showing that the model is learning and improving.

This means the model’s predictions are getting closer to actual values as more data is ingested.

The presence of a smooth downward slope is a strong indicator that the streaming learning process is working correctly.

## Predicted vs True Bitcoin Price
This cell visualizes the **streaming predictions** made by the model and compares them against the **actual observed Bitcoin prices** over time.


In [15]:
def display_pred_true_table(pred_log, true_log):
    import pandas as pd
    df = pd.DataFrame({
        'Time Step': range(1, len(pred_log) + 1),
        'Predicted Price': pred_log,
        'Actual Price': true_log
    })
    print(df.to_string(index=False))

# Example usage
display_pred_true_table(pred_log, true_log)


 Time Step  Predicted Price  Actual Price
         1         0.000000       96855.0
         2      1937.100000       96512.0
         3     20919.189610       96913.0
         4     30328.508092       97017.0
         5     36897.214690       96930.0
         6     29511.930309       96219.0
         7     33634.158659       97026.0
         8     40202.459507       98967.0
         9     66508.370651       99101.0
        10     90549.965374       99532.0
        11    125157.819466      101784.0
        12    135053.147382      101162.0
        13    130326.842781      103076.0
        14    127593.151893      102419.0
        15    124893.544565      103787.0
        16    112008.117990      103000.0
        17    107594.239971      102876.0
        18     98388.220032      103117.0
        19     94792.342875      102963.0
        20     88418.122936      103168.0
        21     89072.551351      103707.0
        22     90183.194471      103479.0
        23     89995.996228      1

The actual price remains relatively stable, reflecting realistic Bitcoin pricing behavior.

The predicted prices initially deviate strongly — showing some overfitting or instability during early updates.

Over time, the predictions begin to stabilize and converge toward the actual values, especially after time step ~15.

This highlights the nature of online learning: the model starts with high error but improves incrementally with each new data point.

we can clearly observe that the model gradually learns to follow the trend of the real prices, though some volatility in prediction remains.

## Print Model Weights
This cell prints the **learned weights** from the linear regression model to understand which features (lags and indicators) are contributing most to predictions.


In [ ]:
# Inspect model weights (feature importance)
print("Model Weights:")
for feature, weight in model[-1].weights.items():
    print(f"{feature}: {weight:.4f}")


## Simulated Retry Logic (API Robustness Test)
This cell simulates an API failure scenario to test the robustness of the retry mechanism implemented in `bitcoin_forecast_utils.py`.


In [ ]:
#  Simulate API failure to demonstrate retry mechanism
def simulate_api_failure():
    raise requests.exceptions.HTTPError(response=requests.Response())

try:
    simulate_api_failure()
except requests.exceptions.HTTPError:
    print("Retry mechanism would be triggered here (simulated).")


##  Save the Trained River Model with Pickle
This cell demonstrates how to serialize and persist the trained River model using the `pickle` module.


In [ ]:
# Save trained River model to a pickle file
with open("btc_stream_model.pkl", "wb") as f:
    pickle.dump(model, f)

print(" Model saved to btc_stream_model.pkl")


## Basic Streaming with OHLC and Linear Regression
This cell demonstrates a simple pipeline for streaming Bitcoin close prices and updating a linear regression model using lag features.

In [ ]:
# 2.12 Basic Streaming with OHLC Close Prices and Linear Regression

ohlc_df = get_coin_ohlc("bitcoin", vs_currency="usd", days=7)
ohlc_prices = ohlc_df["close"].tolist()

rolling_window = deque(maxlen=5)
model = linear_model.LinearRegression()
metric = metrics.MAE()

for price in ohlc_prices[:30]:
    rolling_window.append(price)

    if len(rolling_window) < rolling_window.maxlen:
        continue

    features = build_rolling_features(rolling_window)
    y_pred = model.predict_one(features)
    y_true = features["price_lag_0"]

    if y_pred is not None:
        model.learn_one(features, y_true)
        metric = metric.update(y_true, y_pred)

    rolling_window.append(price)

print(" MAE using OHLC Close Prices:", metric.get())


## Multi-Model Streaming and Volatility Analysis
This cell demonstrates the use of multiple River models—Linear Regression, Hoeffding Tree Regressor, and a Scaled Linear Regression pipeline—to forecast Bitcoin prices using a streaming approach. It also tracks volatility for interpretability.

In [19]:
# ---- Initialization ----
ohlc_df = get_coin_ohlc(days=1)
close_prices = ohlc_df["close"].head(50)  # You can increase this range
rolling_prices = deque(maxlen=5)

# Models
lr_model = linear_model.LinearRegression()
tree_model = tree.HoeffdingTreeRegressor()
pipeline_model = preprocessing.StandardScaler() | linear_model.LinearRegression()

# Logs
actual_log, lr_log, tree_log, pipe_log, vol_log = [], [], [], [], []

# ---- Streaming Loop ----
for step, price in enumerate(close_prices):
    rolling_prices.append(price)
    if len(rolling_prices) < rolling_prices.maxlen:
        continue

    features = build_rolling_features(rolling_prices)
    actual = price

    # Predictions
    lr_pred = lr_model.predict_one(features) if step > rolling_prices.maxlen else 0
    tree_pred = tree_model.predict_one(features) if step > rolling_prices.maxlen else 0
    pipe_pred = pipeline_model.predict_one(features) if step > rolling_prices.maxlen else 0

    # Logging
    actual_log.append(actual)
    lr_log.append(lr_pred)
    tree_log.append(tree_pred)
    pipe_log.append(pipe_pred)
    vol_log.append(ohlc_df["high"].iloc[step] - ohlc_df["low"].iloc[step])

    # Training
    lr_model.learn_one(features, actual)
    tree_model.learn_one(features, actual)
    pipeline_model.learn_one(features, actual)

# ---- Print Output ----
print(f"{'Step':<6} {'Actual':>10} | {'LR':>10} | {'Tree':>10} | {'Pipe':>10} | {'Volatility':>12}")
print("-" * 65)
for i, (a, l, t, p, v) in enumerate(zip(actual_log, lr_log, tree_log, pipe_log, vol_log)):
    print(f"[{i:<3}] {a:10.2f} | {l:10.2f} | {t:10.2f} | {p:10.2f} | {v:12.2f}")

INFO:bitcoin_forecast_utils:Loading cached OHLC data


Step       Actual |         LR |       Tree |       Pipe |   Volatility
-----------------------------------------------------------------
[0  ]   96855.00 |       0.00 |       0.00 |       0.00 |      2311.00
[1  ]   96512.00 |       0.00 |       0.00 |       0.00 |      1189.00
[2  ]   96913.00 | -457522706217716744192.00 |   96683.50 |   20919.19 |       558.00
[3  ]   97017.00 | 3025854411833572352.00 |   96760.00 |   30328.51 |       186.00
[4  ]   96930.00 | -464046551376813817856.00 |   96824.25 |   36897.21 |       649.00
[5  ]   96219.00 | 4898310132939259904.00 |   96845.40 |   29511.93 |       974.00
[6  ]   97026.00 | -463304613120222429184.00 |   96741.00 |   33634.16 |      1267.00
[7  ]   98967.00 | 5417021605459969024.00 |   96781.71 |   40202.46 |      2002.00
[8  ]   99101.00 | -469348801509418336256.00 |   97054.88 |   66508.37 |       717.00
[9  ]   99532.00 | 7521034308172845056.00 |   97282.22 |   90549.97 |       760.00
[10 ]  101784.00 | -479831558025312337920.00

 Prints a neatly formatted row-by-row comparison showing:
  - Time step
  - Actual price
  - Predicted prices from LR, Tree, and Pipeline
  - Rolling volatility

## Visualization: Predicted vs Actual Comparison Across Models
This cell provides a comprehensive visual comparison of the performance of multiple models over time and visualizes the volatility of Bitcoin prices.

 Model Predictions vs Actual Price
- Compares predicted Bitcoin prices from:
  - Linear Regression
  - Hoeffding Tree Regressor
  - Pipeline (StandardScaler + Linear Regression)
- Uses distinct colors and markers for clarity.
- Plots them against the actual price across time steps.

**Insight:** The closer the predicted lines are to the actual line, the better the model’s performance.

 Rolling Volatility (Standard Deviation)
- Displays the rolling volatility of Bitcoin prices (i.e., std deviation of the high-low spread).
- Helps identify periods of high market uncertainty or price fluctuation.

**Insight:** Volatility spikes often indicate sudden price movements which models may struggle to predict accurately.

In [20]:
def display_model_comparison_table(actual_log, lr_log, tree_log, pipe_log):
    import pandas as pd
    df = pd.DataFrame({
        'Time Step': range(1, len(actual_log) + 1),
        'Actual Price': actual_log,
        'Linear Regression': lr_log,
        'Hoeffding Tree': tree_log,
        'Pipeline (Scaled LR)': pipe_log
    })
    print(df.to_string(index=False))

# Example usage
display_model_comparison_table(actual_log, lr_log, tree_log, pipe_log)


 Time Step  Actual Price  Linear Regression  Hoeffding Tree  Pipeline (Scaled LR)
         1       96855.0       0.000000e+00        0.000000              0.000000
         2       96512.0       0.000000e+00        0.000000              0.000000
         3       96913.0      -4.575227e+20    96683.500000          20919.189610
         4       97017.0       3.025854e+18    96760.000000          30328.508092
         5       96930.0      -4.640466e+20    96824.250000          36897.214690
         6       96219.0       4.898310e+18    96845.400000          29511.930309
         7       97026.0      -4.633046e+20    96741.000000          33634.158659
         8       98967.0       5.417022e+18    96781.714286          40202.459507
         9       99101.0      -4.693488e+20    97054.875000          66508.370651
        10       99532.0       7.521034e+18    97282.222222          90549.965374
        11      101784.0      -4.798316e+20    97507.200000         125157.819466
        12      

# Streamlit Integration with Trained River Model
This cell integrates a pre-trained River model into a live Streamlit app for real-time forecasting and dynamic visualization.

In [21]:
# Load trained model
with open("btc_stream_model.pkl", "rb") as f:
    model = pickle.load(f)

# Streamlit UI
st.title(" Real-Time Bitcoin Price Forecasting (River)")

# Fetch price
try:
    current_price = get_bitcoin_price_with_retry()
    st.metric(" Current BTC Price (USD)", f"${current_price:,.2f}")
except Exception as e:
    st.error(f"Failed to fetch price: {e}")
    st.stop()

# Simulate rolling window
if "rolling_prices" not in st.session_state:
    st.session_state.rolling_prices = deque(maxlen=5)

st.session_state.rolling_prices.append(current_price)

# Only forecast if we have enough history
if len(st.session_state.rolling_prices) == 5:
    features = build_rolling_features(st.session_state.rolling_prices)
    prediction = model.predict_one(features)

    # Display prediction
    st.subheader("Predicted Next Price")
    st.success(f"${prediction:,.2f}")

    # Train the model (simulate online learning)
    model.learn_one(features, current_price)

    # Save updated model
    with open("btc_stream_model.pkl", "wb") as f:
        pickle.dump(model, f)

    # Show weights
    st.subheader("Model Weights")
    st.json(model.weights)

    # Plot true vs predicted
    if "price_log" not in st.session_state:
        st.session_state.price_log = []

    st.session_state.price_log.append((current_price, prediction))

    df = pd.DataFrame(st.session_state.price_log, columns=["Actual", "Predicted"])
    st.line_chart(df)

2025-05-15 11:53:53.280 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-15 11:53:53.350 
  command:

    streamlit run /usr/local/lib/python3.8/dist-packages/ipykernel_launcher.py [ARGUMENTS]
2025-05-15 11:53:53.352 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-15 11:53:53.519 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-15 11:53:53.521 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-15 11:53:53.523 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-15 11:53:53.525 Session state does not function when running a script without `streamlit run`
2025-05-15 11:53:53.528 Thread 'MainThread': missing ScriptRunContext! This

In [22]:
code = '''
import streamlit as st
import time
import pickle
from collections import deque
from bitcoin_forecast_utils import get_bitcoin_price_with_retry, build_rolling_features
from river import linear_model, metrics, preprocessing

# App Config
st.set_page_config(page_title="Bitcoin Forecasting with River", page_icon=":chart_with_upwards_trend:")
st.title("📈 Real-Time Bitcoin Price Forecasting using River")

# Load or initialize model with normalization
if "model" not in st.session_state:
    scaler = preprocessing.StandardScaler()
    regressor = linear_model.LinearRegression()
    st.session_state.model = scaler | regressor
    st.session_state.metric = metrics.MAE()
    st.session_state.rolling_prices = deque(maxlen=5)
    st.session_state.price_log = []

model = st.session_state.model
metric = st.session_state.metric
rolling_prices = st.session_state.rolling_prices
price_log = st.session_state.price_log

# Fetch live price
if st.button("🔄 Refresh BTC Price"):
    get_bitcoin_price_with_retry.cache_clear()
    st.rerun()

try:
    current_price = get_bitcoin_price_with_retry()
    st.metric("📌 Current BTC Price (USD)", f"${current_price:,.2f}")
except Exception as e:
    st.error(f"Failed to fetch price: {e}")
    st.stop()

# Update rolling window
rolling_prices.append(current_price)

# Predict only if enough data is available
if len(rolling_prices) == rolling_prices.maxlen:
    features = build_rolling_features(rolling_prices)
    pred_price = model.predict_one(features)

    # Display prediction
    # TEMP FIX: Add predicted delta to current price
    prediction = model.predict_one(features)
    corrected_prediction = current_price + prediction
    st.subheader("🧠 Predicted Next Price")
    st.success(f"${corrected_prediction:,.2f}")


    # Train model
    model.learn_one(features, current_price)
    metric = metric.update(current_price, pred_price)
    st.session_state.metric = metric

    # Log data
    price_log.append((current_price, pred_price))

# Optional: Show model weights
if st.checkbox("🔍 Show Model Weights"):
    try:
        weights = dict(model[-1].weights)
        st.json(weights)
        st.line_chart(list(weights.values()))
    except Exception as e:
        st.error("Could not display weights: " + str(e))

# Optional: Display log chart
if price_log:
    import pandas as pd
    df = pd.DataFrame(price_log, columns=["Actual", "Predicted"])
    st.line_chart(df)

'''

# Save to a file
with open("streamlit_app.py", "w") as f:
    f.write(code)

print(" Streamlit app saved to streamlit_app.py")


 Streamlit app saved to streamlit_app.py


In [ ]:
!streamlit run streamlit_app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.17.0.3:8502
  External URL: http://76.100.194.142:8502

